# 环节 01 · RL 基础与 MDP（配套 Notebook）

> 配套长文：[环节01-RL基础与MDP详解.md](./环节01-RL基础与MDP详解.md)
> 定位：把长文里手推的 Bellman 备份、值迭代收敛、折扣视野跑成可改参数、可复现的代码。全部**纯 Python 标准库**（不用 numpy / torch）。

**怎么跑**

- 依赖：无。逐格 `Shift+Enter`；后面的格子依赖前面已执行的变量。
- 想换 MDP：只改 §1 的 `TRANS`；想换 γ：只改 §2 的 `GAMMA`。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 MDP 定义 | §1 | 五元组怎么落到代码 |
| §2 回报与折扣 | §2 | γ 的"有效视野" = 1/(1−γ) |
| §3 手算最优值 | §4 / §8 | V*(A)=29、V*(B)=30 的方程解 |
| §4 值迭代 | §4 / §8 | 从 0 迭代到不动点，误差每轮 ×γ |
| §5 导出策略 | §4 | π* = argmax Q*，不用单独"训" |
| §6 ε-greedy | §6.1 | 探索-利用权衡 + 纯贪心锁死 |


## 1. MDP 定义：把"试错"写成五元组

```
MDP = (S, A, P, R, γ)

S 状态集合
A 动作集合
P 状态转移 P(s' | s, a)   —— 环境规则
R 奖励     R(s, a, s')    —— 环境的打分
γ 折扣因子  0 ≤ γ ≤ 1
```

下面用一个**只有 2 个状态**的确定性 MDP 贯穿全文（长文 §8 用的同一个）：

| 状态 | 动作 | 转移到 | 奖励 |
|---|---|---|---|
| A | stay | A | +1 |
| A | go | B | +2 |
| B | stay | B | +3 |
| B | go | A | 0 |


In [ ]:
# 确定性 MDP：TRANS[(s, a)] = (s', r)
TRANS = {
    ("A", "stay"): ("A", 1.0),
    ("A", "go"): ("B", 2.0),
    ("B", "stay"): ("B", 3.0),
    ("B", "go"): ("A", 0.0),
}
STATES, ACTIONS, GAMMA = ["A", "B"], ("stay", "go"), 0.9

print("S =", STATES, " A =", list(ACTIONS), " γ =", GAMMA)
for (s, a), (s2, r) in TRANS.items():
    print(f"  {s} --{a:>4}--> {s2},  r = {r:+.0f}")


## 2. 回报与折扣：γ 的"有效视野"

回报 `G_t = r_{t+1} + γ·r_{t+2} + γ²·r_{t+3} + …`

γ 的第一个语义：**有效视野 ≈ 1/(1−γ)**——超过这个步数，奖励的折扣已经小到可以忽略。


In [ ]:
for g in (0.5, 0.9, 0.99, 1.0):
    if g < 1:
        print(f"γ = {g:<5} 有效视野 1/(1-γ) ≈ {1/(1-g):6.1f} 步，γ^50 = {g**50:.2e}")
    else:
        print(f"γ = {g:<5} 无穷和可能发散（须保证回合终止；LLM 用 γ=1 是因为回答必有 EOS）")


## 3. 手算最优值：解 Bellman 最优方程

`V*(s) = max_a [ r + γ·V*(s') ]` 是一个不动点方程。手算：

- B 里 stay 会自我复利：`V*(B) = 3 + 0.9·V*(B)` ⟹ `V*(B) = 3/(1−0.9) = 30`
- A 里跑到 B 最赚：`V*(A) = 2 + 0.9·V*(B) = 29`


In [ ]:
V_B = 3 / (1 - GAMMA)                 # 30：B 一直 stay 的解析解
V_A = 2 + GAMMA * V_B                 # 29：A 走 go 到 B
print(f"解析解：V*(B) = 3/(1-γ)     = {V_B:.1f}")
print(f"        V*(A) = 2 + γ·V*(B) = {V_A:.1f}")
# 代回检查"另一条分支"确实更差 → 说明贪心策略就是最优策略
print(f"检查 A 的 stay 分支：1 + γ·V*(A) = {1 + GAMMA*V_A:6.2f} < {V_A}  → go 更优 ✓")
print(f"检查 B 的 go   分支：0 + γ·V*(A) = {0 + GAMMA*V_A:6.2f} < {V_B}  → stay 更优 ✓")


## 4. 值迭代：从 0 迭代到不动点

反复套 `V ← max_a [r + γV(s')]`。γ<1 时算子是 **γ-压缩映射**（Banach 不动点定理）→ 唯一不动点，且每轮把误差至少乘 γ。


In [ ]:
V = {s: 0.0 for s in STATES}
history = []
for it in range(1, 41):
    V = {s: max(r + GAMMA * V[s2] for (s2, r) in (TRANS[(s, a)] for a in ACTIONS))
         for s in STATES}
    err = max(abs(V[s] - v) for s, v in zip(STATES, (V_A, V_B)))
    history.append(err)
    if it <= 3 or it == 40:
        print(f"第 {it:>2} 轮：V(A) = {V['A']:8.4f}   V(B) = {V['B']:8.4f}   误差 = {err:.4f}")


In [ ]:
# 误差衰减比率实测：每轮把误差至少乘 γ（理论值 0.9）
print("误差衰减比率（理论 γ = 0.9）：")
for it in range(1, 8):
    print(f"  第 {it}→{it+1} 轮：比率 = {history[it] / history[it-1]:.4f}")


## 5. 导出策略：有了 Q\*，策略不需要"训练"

`π*(s) = argmax_a Q*(s, a)` —— 值方法整条路线的世界观（DQN 也建立在这条上）。


In [ ]:
Q_star = {(s, a): r + GAMMA * V[s2] for (s, a), (s2, r) in TRANS.items()}
policy = {s: max(ACTIONS, key=lambda a: Q_star[(s, a)]) for s in STATES}
for s in STATES:
    print(f"  {s}: " + "   ".join(f"{a}: Q* = {Q_star[(s, a)]:7.2f}" for a in ACTIONS))
print("π* =", policy, " （长文手算结论：A→go，B→stay）")


## 6. ε-greedy：探索-利用的权衡

**纯贪心会锁死在次优动作**：某动作第一次试就拿了高分，之后永远只选它，再没机会发现问题。
ε-greedy：以 ε 概率随机、否则贪心。


In [ ]:
EPS = 0.1
n = len(ACTIONS)
print(f"ε = {EPS}，动作数 |A| = {n}")
for a in ACTIONS:
    best = (a == "stay")                      # 假设当前认为 stay 最好
    p = (1 - EPS) + EPS / n if best else EPS / n
    print(f"  π({a:>4}) = {p:.4f}" + ("   ← 当前最优动作，仍以 ε 概率被随机掉" if best else ""))
print(f"→ 最优动作有 {EPS/n:.1%} 的概率不被选中：这就是「探索」的代价。")


## 7. 小结与下钻

- **MDP 是全部算法的公共语言**：S / A / P / R / γ 五元组 + 马尔可夫性。
- **Bellman 把"未来"递归成"一步 + 一个新的未来"**：期望方程评价给定策略，最优方程定义 V*。
- **γ<1 ⟹ 压缩映射 ⟹ 值迭代必收敛**，且每轮误差至少乘 γ。
- **有 Q\* 就不用训策略**：π* 是 Q* 的 argmax。

下一站：[环节 02 · 值方法与 DQN](./环节02-值方法与DQN详解.md)（不知道 P 的时候，怎么把 Q* 逼近出来）。
